# Whisper large-v3-turbo + LoRA for Albanian (Kaggle or Colab) (Gheg + standard)

Fine-tunes `openai/whisper-large-v3-turbo` on Common Voice Albanian (read) and Common Voice
Spontaneous Speech Gheg, then measures base vs fine-tuned on the **held-out test speakers**
(never used in training), and exports a CTranslate2 model for faster-whisper / SyncUp.

**Kaggle:** Settings → Accelerator **GPU** · Internet **On** · Add Input → the private dataset with
the two `.tar.gz` files. Output: `whisper-turbo-sq-ct2.zip` in the Output tab.

**Google Colab:** Runtime → Change runtime type → **T4 GPU**. Put the two `.tar.gz` files in Google Drive,
folder `MyDrive/albanian-asr`, and allow Drive access when asked. Output: `MyDrive/albanian-asr/output`.
If Colab disconnects: run the cells up to Train again, then `trainer.train(resume_from_checkpoint=True)`.

Takes roughly 2–4 hours. Only public CC0 data belongs here: company meeting recordings are never
uploaded to Kaggle, Colab or Drive.

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"          # one GPU: LoRA + 2 GPUs (DataParallel) is fragile
!pip install -q peft jiwer ctranslate2
!pip uninstall -y -q torchao   # Kaggle's old torchao breaks peft; LoRA here does not use it
import torch

# Kaggle (needs phone verification for a GPU) or Google Colab (Google login only)
if os.path.exists("/kaggle"):
    INPUTS, WORK, TMP = ["/kaggle/input"], "/kaggle/working", "/kaggle/temp/data"
else:  # Colab: put the two .tar.gz files in Google Drive, folder MyDrive/albanian-asr
    from google.colab import drive
    drive.mount("/content/drive")
    INPUTS, WORK, TMP = ["/content/drive/MyDrive/albanian-asr"], "/content/drive/MyDrive/albanian-asr/output", "/content/data"
os.makedirs(WORK, exist_ok=True)   # on Colab this is on Drive: checkpoints survive a disconnect
print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU")
assert torch.cuda.is_available(), "Turn on the GPU: Settings -> Accelerator"

## 1. Find and unpack the data

In [ ]:
import glob, tarfile
from pathlib import Path

def find(name):
    for root in [TMP] + INPUTS:
        hits = glob.glob(f"{root}/**/{name}", recursive=True)
        if hits:
            return Path(hits[0])
    return None

if not find("ss-corpus-aln.tsv") or not find("validated.tsv"):
    out = Path(TMP); out.mkdir(parents=True, exist_ok=True)
    tgzs = [t for root in INPUTS for t in glob.glob(f"{root}/**/*.tar.gz", recursive=True)]
    assert tgzs, f"The two .tar.gz files were not found in {INPUTS}"
    for tgz in tgzs:
        print("unpacking", tgz)
        with tarfile.open(tgz) as t:
            t.extractall(out, filter="data")

GHEG_TSV, SQ_TSV, DUR_TSV = find("ss-corpus-aln.tsv"), find("validated.tsv"), find("clip_durations.tsv")
# Kaggle unpacks uploaded archives itself and may spread one archive over several folders,
# so every audio file is looked up by its name, wherever it landed.
MP3 = {}
for root in [TMP] + INPUTS:
    for folder, _, files in os.walk(root):
        for f in files:
            if f.endswith(".mp3"):
                MP3.setdefault(f, os.path.join(folder, f))
print("Gheg list:", GHEG_TSV); print("Albanian list:", SQ_TSV); print("Durations:", DUR_TSV)
print(len(MP3), "audio files found")

## 2. Build train / validation / test (speaker-disjoint)

* **Gheg:** the dataset's own splits (train 7 speakers, dev 3, test 3; no speaker in two splits).
* **Albanian (read):** own speaker-disjoint split of the validated clips (Common Voice's split has only 2 training speakers); test sentences never appear in training.
* The Gheg transcriber wrote `w` for ë and `C` for ç (see its README): converted to normal letters.
* Clips longer than 30 s are left out (Whisper reads 30-second windows).

In [ ]:
import re
import pandas as pd

def gheg_letters(t):
    t = str(t).replace("w", "ë").replace("W", "Ë").replace("C", "ç")
    t = re.sub(r"\[\?+\]", " ", t)
    return re.sub(r"\s+", " ", t.replace("[", "").replace("]", "")).strip()

g = pd.read_csv(GHEG_TSV, sep="\t")
g = g[g.transcription.notna() & g.split.isin(["train", "dev", "test"]) & (g.duration_ms <= 30000)]
gheg = pd.DataFrame({"path": g.audio_file.map(MP3), "text": g.transcription.map(gheg_letters),
                     "speaker": g.client_id, "split": g.split.replace({"dev": "validation"}), "source": "gheg"})

val = pd.read_csv(SQ_TSV, sep="\t", quoting=3)
# clip lengths: from clip_durations.tsv when it is there (read clips are ~5 s, so without it nothing is left out)
dur = (pd.read_csv(DUR_TSV, sep="\t").set_index("clip").iloc[:, 0] if DUR_TSV
       else pd.Series(0, index=val.path))
# Common Voice's own split puts 138 of 147 Albanian speakers in test and 2 in train: useless for
# training. Own speaker-disjoint split instead (same rule as scripts/asr_benchmark.py):
# among speakers with <= 100 clips, sorted by id, every 4th is test, every 8th (offset 1) validation.
counts = val.client_id.value_counts()
small = sorted(counts[counts <= 100].index)
test_spk, val_spk = set(small[::4]), set(small[1::8])
test, test_dev = val[val.client_id.isin(test_spk)], val[val.client_id.isin(val_spk)]
held_txt = set(test.sentence) | set(test_dev.sentence)
train_sq = val[~val.client_id.isin(test_spk | val_spk) & ~val.sentence.isin(held_txt)]
def sq_rows(df, split):
    df = df[df.path.map(dur).fillna(0) <= 30000]
    return pd.DataFrame({"path": df.path.map(MP3), "text": df.sentence.str.strip(),
                         "speaker": df.client_id, "split": split, "source": "sq"})
sq = pd.concat([sq_rows(train_sq, "train"), sq_rows(test_dev, "validation"), sq_rows(test, "test")])

data = pd.concat([gheg, sq], ignore_index=True)
missing = data.path.isna().sum()
print(f"{missing} clips without an audio file are left out")
data = data[data.path.notna() & (data.text.str.len() > 0)]
for a, b in [("train", "test"), ("train", "validation")]:
    assert not set(data[data.split == a].speaker) & set(data[data.split == b].speaker), f"speaker in {a} and {b}"
print(data.groupby(["source", "split"]).agg(clips=("path", "count"), speakers=("speaker", "nunique")))

## 3. Model, processor and LoRA

In [ ]:
from transformers import WhisperProcessor, WhisperForConditionalGeneration
from peft import LoraConfig, get_peft_model

BASE = "openai/whisper-large-v3-turbo"
processor = WhisperProcessor.from_pretrained(BASE, language="albanian", task="transcribe")
model = WhisperForConditionalGeneration.from_pretrained(BASE)
model.config.use_cache = False
model.generation_config.language = "albanian"
model.generation_config.task = "transcribe"
model.generation_config.forced_decoder_ids = None

lora = LoraConfig(r=32, lora_alpha=64, lora_dropout=0.05,
                  target_modules=["q_proj", "k_proj", "v_proj", "out_proj", "fc1", "fc2"])
model = get_peft_model(model, lora)
model.print_trainable_parameters()

## 4. Audio is read on the fly (the features of every clip would not fit in memory)

In [ ]:
import librosa, torch
from torch.utils.data import Dataset

class Clips(Dataset):
    def __init__(self, df): self.df = df.reset_index(drop=True)
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        r = self.df.iloc[i]
        audio, _ = librosa.load(r.path, sr=16000, mono=True)
        feats = processor.feature_extractor(audio, sampling_rate=16000).input_features[0]
        labels = processor.tokenizer(r.text).input_ids[:440]
        return {"input_features": feats, "labels": labels}

class Collator:
    def __init__(self, start_id): self.start_id = start_id
    def __call__(self, feats):
        batch = processor.feature_extractor.pad([{"input_features": f["input_features"]} for f in feats], return_tensors="pt")
        lab = processor.tokenizer.pad([{"input_ids": f["labels"]} for f in feats], return_tensors="pt")
        labels = lab["input_ids"].masked_fill(lab.attention_mask.ne(1), -100)
        if (labels[:, 0] == self.start_id).all():
            labels = labels[:, 1:]
        batch["labels"] = labels
        return batch

train_ds = Clips(data[data.split == "train"].sample(frac=1, random_state=42))
# validation: all Gheg dev + 300 read clips, so one evaluation stays a few minutes
v = data[data.split == "validation"]
val_ds = Clips(pd.concat([v[v.source == "gheg"], v[v.source == "sq"].sample(min(300, (v.source == "sq").sum()), random_state=1)]))
print(len(train_ds), "train clips,", len(val_ds), "validation clips")

## 5. Train (checkpoints in WORK/out, the best one kept)

In [ ]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer

args = Seq2SeqTrainingArguments(
    output_dir=f"{WORK}/out",
    per_device_train_batch_size=8, gradient_accumulation_steps=4,   # 32 clips per step
    per_device_eval_batch_size=8,
    learning_rate=1e-4, warmup_steps=100, num_train_epochs=4, lr_scheduler_type="linear",
    fp16=True, gradient_checkpointing=True, gradient_checkpointing_kwargs={"use_reentrant": False},
    eval_strategy="steps", eval_steps=150, save_steps=150, save_total_limit=2,
    load_best_model_at_end=True, metric_for_best_model="eval_loss", greater_is_better=False,
    logging_steps=25, dataloader_num_workers=2, remove_unused_columns=False,
    label_names=["labels"], report_to="none")

trainer = Seq2SeqTrainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
                         data_collator=Collator(model.config.decoder_start_token_id))
trainer.train()   # after a cut: run the cells above again, then trainer.train(resume_from_checkpoint=True)

## 6. Base vs fine-tuned on the held-out test speakers

Same clips, same decoding for both models. WER = word error rate, CER = character error rate,
after lower-casing and removing punctuation.

In [ ]:
import unicodedata, jiwer, numpy as np

def norm(t):
    t = unicodedata.normalize("NFC", str(t)).lower()
    t = re.sub(r"[-–—/]", " ", t)
    return re.sub(r"\s+", " ", re.sub(r"[^\w\s]", "", t)).strip()

def transcribe(m, paths, bs=8):
    m.eval(); out = []
    for i in range(0, len(paths), bs):
        audio = [librosa.load(p, sr=16000)[0] for p in paths[i:i + bs]]
        f = processor.feature_extractor(audio, sampling_rate=16000, return_tensors="pt").input_features
        with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):
            ids = m.generate(input_features=f.to("cuda"), language="albanian", task="transcribe", max_new_tokens=440, num_beams=1)
        out += processor.batch_decode(ids, skip_special_tokens=True)
    return out

def score(df, hyps):
    refs, hyps = [norm(x) for x in df.text], [norm(x) for x in hyps]
    return jiwer.wer(refs, hyps), jiwer.cer(refs, hyps)

test_sets = {"Gheg spontaneous (3 speakers)": data[(data.split == "test") & (data.source == "gheg")],
             "Albanian read": data[(data.split == "test") & (data.source == "sq")]}

model.config.use_cache = True
rows = []
for name, df in test_sets.items():
    tuned = transcribe(model, list(df.path))
    with model.disable_adapter():
        base = transcribe(model, list(df.path))
    (bw, bc), (tw, tc) = score(df, base), score(df, tuned)
    rows.append({"test set": name, "clips": len(df), "base WER": f"{bw:.1%}", "fine-tuned WER": f"{tw:.1%}",
                 "base CER": f"{bc:.1%}", "fine-tuned CER": f"{tc:.1%}"})
    pd.DataFrame({"ref": df.text.values, "base": base, "tuned": tuned}).to_csv(
        f"{WORK}/test_{name.split()[0].lower()}.csv", index=False)
results = pd.DataFrame(rows); results.to_csv(f"{WORK}/results.csv", index=False)
results

## 7. Save, convert for faster-whisper, zip for download

In [ ]:
import json, shutil
from huggingface_hub import hf_hub_download
from ctranslate2.converters import TransformersConverter

merged = model.merge_and_unload()
merged = merged.half()
merged.save_pretrained(f"{WORK}/whisper-turbo-sq")
processor.save_pretrained(f"{WORK}/whisper-turbo-sq")
for f in ("tokenizer.json", "preprocessor_config.json"):   # the converter copies these
    if not os.path.exists(f"{WORK}/whisper-turbo-sq/{f}"):
        shutil.copy(hf_hub_download(BASE, f), f"{WORK}/whisper-turbo-sq/{f}")
json.dump({"base": BASE, "lora": lora.to_dict() if hasattr(lora, "to_dict") else str(lora),
           "train_clips": len(train_ds), "results": rows, "log": trainer.state.log_history},
          open(f"{WORK}/whisper-turbo-sq/training_info.json", "w"), indent=1, default=str)

TransformersConverter(f"{WORK}/whisper-turbo-sq",
                      copy_files=["tokenizer.json", "preprocessor_config.json"]).convert(
    f"{WORK}/whisper-turbo-sq-ct2", quantization="int8", force=True)
shutil.copy(f"{WORK}/whisper-turbo-sq/training_info.json", f"{WORK}/whisper-turbo-sq-ct2/")
shutil.make_archive(f"{WORK}/whisper-turbo-sq-ct2", "zip", f"{WORK}/whisper-turbo-sq-ct2")
shutil.rmtree(f"{WORK}/out", ignore_errors=True)       # checkpoints: no longer needed
print("Done. Download whisper-turbo-sq-ct2.zip from the Output tab.")